In [5]:
"""
msa_gva_ranks.py
================
Reads ONS Regional GVA per head (Table 2) and aggregates
to MSA level using the official ONS LAD → ITL3 lookup.
"""

import pandas as pd
import numpy as np
# /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map/data/raw/gva/gva-by-industry-by-local-authority-time-series-v1.csv
BASE     = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map"
GVA_PATH = f"{BASE}/data/raw/gva/regionalgrossvalueaddedbalancedperheadandincomecomponents.xlsx"
ITL_LKP  = f"{BASE}/data/raw/lookups/LAD_(December_2024)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2025)_Lookup_in_the_UK.csv"
POP_CSV  = f"{BASE}/data/processed/msa_population_ranks.csv"

# ── 1. Load ITL lookup ────────────────────────────────────────────────────
print("── 1. Loading LAD → ITL lookup ──────────────────────────────────")
itl_lkp = pd.read_csv(ITL_LKP)
print(f"   Rows: {len(itl_lkp)}")
print(f"   Columns: {list(itl_lkp.columns)}")
print(f"\n   First 3 rows:")
print(itl_lkp.head(3).to_string())

── 1. Loading LAD → ITL lookup ──────────────────────────────────
   Rows: 362
   Columns: ['ITL125CD', 'ITL125NM', 'ITL225CD', 'ITL225NM', 'ITL325CD', 'ITL325NM', 'LAU125CD', 'LAU125NM', 'LAD24CD', 'LAD24NM', 'ObjectId']

   First 3 rows:
  ITL125CD              ITL125NM ITL225CD     ITL225NM ITL325CD                         ITL325NM   LAU125CD          LAU125NM    LAD24CD           LAD24NM  ObjectId
0      TLC  North East (England)     TLC3  Tees Valley    TLC31  Hartlepool and Stockton-on-Tees  E06000001        Hartlepool  E06000001        Hartlepool         1
1      TLC  North East (England)     TLC3  Tees Valley    TLC31  Hartlepool and Stockton-on-Tees  E06000004  Stockton-on-Tees  E06000004  Stockton-on-Tees         2
2      TLC  North East (England)     TLC3  Tees Valley    TLC32                   South Teesside  E06000002     Middlesbrough  E06000002     Middlesbrough         3


In [6]:
"""
msa_gva_ranks.py
================
Reads ONS Regional GVA per head (Table 2) and aggregates
to MSA level using the official ONS LAD → ITL3 lookup.

Population-weighted average used where an MSA spans
multiple ITL3 areas with different populations.
"""

import pandas as pd
import numpy as np

# BASE     = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"
GVA_PATH = f"{BASE}/data/raw/gva/regionalgrossvalueaddedbalancedperheadandincomecomponents.xlsx"
ITL_LKP  = f"{BASE}/data/raw/lookups/LAD_(December_2024)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2025)_Lookup_in_the_UK.csv"
POP_CSV  = f"{BASE}/data/processed/msa_population_ranks.csv"

LATEST_YEAR = "2023"

# ── 1. Load ITL lookup ────────────────────────────────────────────────────
print("── 1. Loading LAD → ITL lookup ──────────────────────────────────")
itl_lkp = pd.read_csv(ITL_LKP)[["LAD24CD","ITL325CD","ITL325NM","ITL225CD","ITL225NM"]].copy()
print(f"   Rows: {len(itl_lkp)}  |  ITL3 areas: {itl_lkp['ITL325CD'].nunique()}")

# ── 2. Load GVA per head Table 2 ─────────────────────────────────────────
print("\n── 2. Loading GVA per head Table 2 ─────────────────────────────")
raw = pd.read_excel(GVA_PATH, sheet_name="Table 2", header=1)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={
    "ITL":         "itl_level",
    "ITL code":    "itl_code",
    "Region name": "region_name",
})
raw = raw.dropna(subset=["itl_code"])
raw["itl_code"]  = raw["itl_code"].astype(str).str.strip()
raw["itl_level"] = raw["itl_level"].astype(str).str.strip()

# Extract benchmark rows
uk_gva  = float(raw[raw["itl_code"] == "UK"][LATEST_YEAR].iloc[0])
eng_gva = float(raw[raw["itl_code"] == "TLB"][LATEST_YEAR].iloc[0])

# Keep ITL3 only
itl3 = raw[raw["itl_level"] == "ITL3"][["itl_code","region_name"] +
          [str(y) for y in range(1998, 2024)]].copy()

# Convert year columns to numeric
for y in range(1998, 2024):
    itl3[str(y)] = pd.to_numeric(itl3[str(y)], errors="coerce")

print(f"   ITL3 areas in GVA file: {len(itl3)}")
print(f"   UK  GVA per head {LATEST_YEAR}: £{uk_gva:,.0f}")
print(f"   Eng GVA per head {LATEST_YEAR}: £{eng_gva:,.0f}")

# ── 3. Load population for weighting ─────────────────────────────────────
# Need ITL3-level population to weight per head figures when combining areas
# Use LA populations from MYE, aggregate to ITL3 via lookup
print("\n── 3. Building ITL3 population weights ──────────────────────────")

# Load MYE LA populations
mye_path = f"{BASE}/data/raw/population/mye24tablesew.xlsx"
mye = pd.read_excel(mye_path, sheet_name="MYE5", header=7)
mye.columns = mye.columns.str.strip()
mye = mye.rename(columns={
    "Code":                          "la_code",
    "Estimated Population mid-2023": "pop_2023",
    "Estimated Population mid-2022": "pop_2022",
    "Estimated Population mid-2021": "pop_2021",
})
mye = mye.dropna(subset=["la_code"])
mye = mye[mye["la_code"].astype(str).str.match(r"^[EW]\d+")]
mye["la_code"]   = mye["la_code"].astype(str)
mye["pop_2023"]  = pd.to_numeric(mye["pop_2023"], errors="coerce")

# Join LA pop → ITL3 via lookup
la_itl3 = itl_lkp.merge(
    mye[["la_code","pop_2023"]],
    left_on="LAD24CD", right_on="la_code", how="left"
)

# Aggregate LA pops to ITL3 level
itl3_pop = la_itl3.groupby(["ITL325CD","ITL325NM"])["pop_2023"].sum().reset_index()
itl3_pop.columns = ["itl_code","itl_name","pop_2023"]
print(f"   ITL3 areas with population: {len(itl3_pop)}")

# Join population onto GVA
itl3 = itl3.merge(itl3_pop[["itl_code","pop_2023"]], on="itl_code", how="left")
missing_pop = itl3["pop_2023"].isna().sum()
if missing_pop > 0:
    print(f"   ⚠  {missing_pop} ITL3 areas missing population — check codes")

# ── 4. Define MSA constituent LAD codes ───────────────────────────────────
print("\n── 4. Mapping MSA LADs → ITL3 ───────────────────────────────────")

MSA_LADS = {
    "Greater Manchester":            ["E08000001","E08000002","E08000003","E08000004",
                                      "E08000005","E08000006","E08000007","E08000008",
                                      "E08000009","E08000010"],
    "West Midlands":                 ["E08000025","E08000026","E08000027","E08000028",
                                      "E08000029","E08000030","E08000031"],
    "North East":                    ["E06000047","E06000005","E06000001","E06000002",
                                      "E06000057","E06000003","E06000004",
                                      "E08000037","E08000021","E08000022",
                                      "E08000023","E08000024"],
    "Liverpool City Region":         ["E08000011","E08000012","E08000014",
                                      "E08000013","E08000015","E06000006"],
    "South Yorkshire":               ["E08000016","E08000017","E08000018","E08000019"],
    "West Yorkshire":                ["E08000032","E08000033","E08000034",
                                      "E08000035","E08000036"],
    "West of England":               ["E06000025","E06000022","E06000023","E06000024"],
    "Lancashire":                    ["E07000117","E07000118","E07000119","E07000120",
                                      "E07000121","E07000122","E07000123","E07000124",
                                      "E07000125","E07000126","E07000127","E07000128",
                                      "E06000008","E06000009"],
    "Tees Valley":                   ["E06000001","E06000002","E06000003",
                                      "E06000004","E06000005"],
    "York & North Yorkshire":        ["E06000014","E06000065"],
    "Hull & East Yorkshire":         ["E06000010","E06000011"],
    "Greater Lincolnshire":          ["E06000012","E06000013","E07000136","E07000137",
                                      "E07000138","E07000139","E07000140","E07000141",
                                      "E07000142"],
    "East Midlands":                 ["E06000015","E06000016","E06000017","E06000018",
                                      "E07000129","E07000130","E07000131","E07000132",
                                      "E07000133","E07000134","E07000135",
                                      "E07000170","E07000171","E07000172","E07000173",
                                      "E07000174","E07000175","E07000176",
                                      "E07000177","E07000178","E07000179","E07000180",
                                      "E07000181"],
    "Cambridgeshire & Peterborough": ["E06000031","E07000008","E07000009",
                                      "E07000010","E07000011","E07000012"],
    "Sussex & Brighton":             ["E07000061","E07000062","E07000063",
                                      "E07000064","E07000065",
                                      "E07000223","E07000224","E07000225",
                                      "E07000226","E07000227","E07000228","E07000229",
                                      "E06000043"],
    "Hampshire & Solent":            ["E07000084","E07000085","E07000086","E07000087",
                                      "E07000088","E07000089","E07000090","E07000091",
                                      "E07000092","E07000093","E07000094",
                                      "E06000044","E06000045"],
    "Cheshire & Warrington":         ["E06000049","E06000050","E06000007"],
    "Cumbria":                       ["E06000063","E06000064"],
    "Greater Essex":                 ["E07000066","E07000067","E07000068","E07000069",
                                      "E07000070","E07000071","E07000072","E07000073",
                                      "E07000074","E07000075","E07000076","E07000077",
                                      "E06000033","E06000034"],
    "Norfolk & Suffolk":             ["E07000143","E07000144","E07000145","E07000146",
                                      "E07000147","E07000148","E07000149",
                                      "E07000200","E07000202","E07000203",
                                      "E07000244","E07000245"],
}

MSA_TIERS = {
    "Greater Manchester":            "Established",
    "West Midlands":                 "Established",
    "North East":                    "Established",
    "Liverpool City Region":         "Established",
    "South Yorkshire":               "Established",
    "West Yorkshire":                "Established",
    "West of England":               "Established",
    "Lancashire":                    "Established",
    "Tees Valley":                   "Existing",
    "York & North Yorkshire":        "Existing",
    "Hull & East Yorkshire":         "Existing",
    "Greater Lincolnshire":          "Existing",
    "East Midlands":                 "Existing",
    "Cambridgeshire & Peterborough": "Existing",
    "Sussex & Brighton":             "DPP",
    "Hampshire & Solent":            "DPP",
    "Cheshire & Warrington":         "DPP",
    "Cumbria":                       "DPP",
    "Greater Essex":                 "DPP",
    "Norfolk & Suffolk":             "DPP",
}

# Map each MSA's LADs → unique ITL3 codes
msa_itl3_map = {}
for msa, lad_codes in MSA_LADS.items():
    matched = itl_lkp[itl_lkp["LAD24CD"].isin(lad_codes)]
    uniq_itl3 = matched[["ITL325CD","ITL325NM"]].drop_duplicates()
    msa_itl3_map[msa] = uniq_itl3
    n_lads   = len(matched)
    n_itl3   = len(uniq_itl3)
    missing  = [c for c in lad_codes if c not in itl_lkp["LAD24CD"].values]
    status   = f"⚠  {len(missing)} LADs not in lookup: {missing}" if missing else "✓"
    print(f"   {status}  {msa:<36} {n_lads} LADs → {n_itl3} ITL3 areas: "
          f"{uniq_itl3['ITL325CD'].tolist()}")

# ── 5. Aggregate GVA per head to MSA level ────────────────────────────────
print("\n── 5. Aggregating GVA per head to MSA level ─────────────────────")

YEARS = [str(y) for y in range(1998, 2024)]

rows = []
for msa, itl3_codes_df in msa_itl3_map.items():
    codes  = itl3_codes_df["ITL325CD"].tolist()
    subset = itl3.merge(itl3_codes_df, left_on="itl_code", right_on="ITL325CD", how="inner")

    if subset.empty:
        print(f"   ⚠  No GVA data for {msa} — codes: {codes}")
        continue

    row = {"msa_name": msa, "tier": MSA_TIERS[msa], "n_itl3": len(subset)}

    if len(subset) == 1:
        # Single ITL3 — use directly
        for y in YEARS:
            row[f"gva_ph_{y}"] = float(subset[y].iloc[0])
        row["method"] = "direct"
    else:
        # Multiple ITL3 areas — population-weighted average
        total_pop = subset["pop_2023"].sum()
        if total_pop == 0 or subset["pop_2023"].isna().all():
            # Fall back to simple average if no population data
            for y in YEARS:
                row[f"gva_ph_{y}"] = float(subset[y].mean())
            row["method"] = "simple_avg (no pop)"
        else:
            subset = subset.copy()
            subset["weight"] = subset["pop_2023"] / total_pop
            for y in YEARS:
                vals = pd.to_numeric(subset[y], errors="coerce")
                row[f"gva_ph_{y}"] = float((vals * subset["weight"]).sum())
            row["method"] = f"pop_weighted ({len(subset)} ITL3)"

    rows.append(row)

msa_df = pd.DataFrame(rows)

# ── 6. Compute derived metrics ────────────────────────────────────────────
print("── 6. Computing derived metrics ─────────────────────────────────")

msa_df["gva_ph_2023"]        = msa_df["gva_ph_2023"].round(0)
msa_df["gva_ph_2019"]        = msa_df["gva_ph_2019"].round(0)
msa_df["gva_ph_2013"]        = msa_df["gva_ph_2013"].round(0)

# Growth rates
msa_df["gva_growth_2019_23"] = ((msa_df["gva_ph_2023"] / msa_df["gva_ph_2019"]) - 1).round(4) * 100
msa_df["gva_growth_2013_23"] = ((msa_df["gva_ph_2023"] / msa_df["gva_ph_2013"]) - 1).round(4) * 100
msa_df["gva_growth_2018_23"] = ((msa_df["gva_ph_2023"] / msa_df["gva_ph_2018"]) - 1).round(4) * 100
msa_df["rank_gva_growth_1823"] = msa_df["gva_growth_2018_23"].rank(ascending=False, method="min").astype(int)

# Index vs England (England = 100)
msa_df["gva_index_eng"]      = (msa_df["gva_ph_2023"] / eng_gva * 100).round(1)

# Rankings (higher GVA per head = rank 1)
msa_df["rank_gva_ph"]        = msa_df["gva_ph_2023"].rank(ascending=False, method="min").astype(int)
msa_df["rank_gva_growth"]    = msa_df["gva_growth_2019_23"].rank(ascending=False, method="min").astype(int)

# ── 7. England benchmarks ─────────────────────────────────────────────────
eng_gva_2019   = float(raw[raw["itl_code"] == "TLB"]["2019"].iloc[0])
eng_gva_2013   = float(raw[raw["itl_code"] == "TLB"]["2013"].iloc[0])
eng_growth     = round((eng_gva / eng_gva_2019 - 1) * 100, 2)
eng_growth_10y = round((eng_gva / eng_gva_2013 - 1) * 100, 2)

# ── 8. Print results ──────────────────────────────────────────────────────
print("\n── GVA per head results (sorted highest to lowest) ──────────────")
print(f"\n{'MSA':<36} {'Tier':<12} {'GVA/head':>10} {'Rank':>5} "
      f"{'Eng=100':>8} {'Growth 19-23':>13} {'Rank':>5} {'Method'}")
print("-" * 110)

for _, r in msa_df.sort_values("gva_ph_2023", ascending=False).iterrows():
    print(
        f"{r['msa_name']:<36} "
        f"{r['tier']:<12} "
        f"£{r['gva_ph_2023']:>9,.0f} "
        f"{r['rank_gva_ph']:>5} "
        f"{r['gva_index_eng']:>7.1f} "
        f"{r['gva_growth_2019_23']:>12.1f}% "
        f"{r['rank_gva_growth']:>5}  "
        f"{r['method']}"
    )

print(f"\n{'England (benchmark)':<36} {'':12} £{eng_gva:>9,.0f} "
      f"{'—':>5} {'100.0':>7} {eng_growth:>12.1f}% {'—':>5}")

# ── 9. YNYCA spotlight ────────────────────────────────────────────────────
print("\n── YNYCA spotlight ──────────────────────────────────────────────")
y = msa_df[msa_df["msa_name"] == "York & North Yorkshire"].iloc[0]
print(f"  GVA per head 2023:      £{y['gva_ph_2023']:,.0f}  "
      f"(rank {y['rank_gva_ph']}/20, England index {y['gva_index_eng']})")
print(f"  GVA per head 2019:      £{y['gva_ph_2019']:,.0f}")
print(f"  Growth 2019-2023:        {y['gva_growth_2019_23']:.1f}%  "
      f"(rank {y['rank_gva_growth']}/20, England: {eng_growth:.1f}%)")
print(f"  Growth 2013-2023:        {y['gva_growth_2013_23']:.1f}%  "
      f"(England: {eng_growth_10y:.1f}%)")
print(f"  ITL3 method:            {y['method']}")

# ── 10. Save ──────────────────────────────────────────────────────────────
out = f"{BASE}/data/processed/msa_gva_ranks.csv"
msa_df.to_csv(out, index=False)
print(f"\n── Saved: {out}")

── 1. Loading LAD → ITL lookup ──────────────────────────────────
   Rows: 362  |  ITL3 areas: 182

── 2. Loading GVA per head Table 2 ─────────────────────────────
   ITL3 areas in GVA file: 182
   UK  GVA per head 2023: £36,103
   Eng GVA per head 2023: £36,632

── 3. Building ITL3 population weights ──────────────────────────
   ITL3 areas with population: 182

── 4. Mapping MSA LADs → ITL3 ───────────────────────────────────
   ✓  Greater Manchester                   10 LADs → 5 ITL3 areas: ['TLD33', 'TLD34', 'TLD35', 'TLD36', 'TLD37']
   ✓  West Midlands                        7 LADs → 7 ITL3 areas: ['TLG31', 'TLG32', 'TLG33', 'TLG36', 'TLG37', 'TLG38', 'TLG39']
   ✓  North East                           12 LADs → 7 ITL3 areas: ['TLC31', 'TLC32', 'TLC33', 'TLC41', 'TLC42', 'TLC43', 'TLC44']
   ✓  Liverpool City Region                6 LADs → 4 ITL3 areas: ['TLD71', 'TLD72', 'TLD73', 'TLD74']
   ✓  South Yorkshire                      4 LADs → 4 ITL3 areas: ['TLE32', 'TLE33', 'TLE3

In [7]:
# For each MSA, show which LADs are inside vs outside
# and whether any ITL3 areas are shared with non-MSA LADs
print("── ITL3 boundary check ──────────────────────────────────────────")

all_msa_lads = set()
for lads in MSA_LADS.values():
    all_msa_lads.update(lads)

for msa, itl3_df in msa_itl3_map.items():
    codes = itl3_df["ITL325CD"].tolist()
    # All LADs in these ITL3 areas
    all_lads_in_itl3 = itl_lkp[itl_lkp["ITL325CD"].isin(codes)]["LAD24CD"].unique()
    # LADs in these ITL3 areas but NOT in this MSA
    msa_lads = set(MSA_LADS[msa])
    outside_lads = [c for c in all_lads_in_itl3 if c not in msa_lads]
    
    if outside_lads:
        outside_names = itl_lkp[itl_lkp["LAD24CD"].isin(outside_lads)]["LAD24NM"].unique()
        print(f"\n  ⚠  {msa}")
        print(f"     ITL3 areas include {len(outside_lads)} LADs outside MSA boundary:")
        for n in outside_names[:8]:
            print(f"       - {n}")
        if len(outside_names) > 8:
            print(f"       ... and {len(outside_names)-8} more")
    else:
        print(f"  ✓  {msa} — ITL3 boundaries match MSA exactly")

── ITL3 boundary check ──────────────────────────────────────────
  ✓  Greater Manchester — ITL3 boundaries match MSA exactly
  ✓  West Midlands — ITL3 boundaries match MSA exactly
  ✓  North East — ITL3 boundaries match MSA exactly
  ✓  Liverpool City Region — ITL3 boundaries match MSA exactly
  ✓  South Yorkshire — ITL3 boundaries match MSA exactly
  ✓  West Yorkshire — ITL3 boundaries match MSA exactly
  ✓  West of England — ITL3 boundaries match MSA exactly
  ✓  Lancashire — ITL3 boundaries match MSA exactly
  ✓  Tees Valley — ITL3 boundaries match MSA exactly
  ✓  York & North Yorkshire — ITL3 boundaries match MSA exactly
  ✓  Hull & East Yorkshire — ITL3 boundaries match MSA exactly
  ✓  Greater Lincolnshire — ITL3 boundaries match MSA exactly
  ✓  East Midlands — ITL3 boundaries match MSA exactly
  ✓  Cambridgeshire & Peterborough — ITL3 boundaries match MSA exactly
  ✓  Sussex & Brighton — ITL3 boundaries match MSA exactly
  ✓  Hampshire & Solent — ITL3 boundaries match MSA exa

In [8]:
# Sense check: compare growth measures avoiding COVID trough
msa_df["gva_growth_2018_23"] = ((msa_df["gva_ph_2023"] / msa_df["gva_ph_2018"]) - 1).round(4) * 100
msa_df["rank_gva_growth_1823"] = msa_df["gva_growth_2018_23"].rank(ascending=False, method="min").astype(int)

eng_gva_2018 = float(raw[raw["itl_code"] == "TLB"]["2018"].iloc[0])
eng_growth_1823 = round((eng_gva / eng_gva_2018 - 1) * 100, 2)

print("\n── Growth sense check (2018-2023, avoids COVID trough) ──────────")
print(f"\n{'MSA':<36} {'GVA 2023':>10} {'Growth 19-23':>13} {'Growth 18-23':>13}")
print("-" * 75)
for _, r in msa_df.sort_values("gva_ph_2023", ascending=False).iterrows():
    print(f"{r['msa_name']:<36} £{r['gva_ph_2023']:>9,.0f} "
          f"{r['gva_growth_2019_23']:>12.1f}% "
          f"{r['gva_growth_2018_23']:>12.1f}%")
print(f"\n{'England':<36} £{eng_gva:>9,.0f} "
      f"{eng_growth:>12.1f}% "
      f"{eng_growth_1823:>12.1f}%")


── Growth sense check (2018-2023, avoids COVID trough) ──────────

MSA                                    GVA 2023  Growth 19-23  Growth 18-23
---------------------------------------------------------------------------
Cheshire & Warrington                £   42,015         19.3%         24.5%
West of England                      £   40,793         28.4%         34.1%
Cambridgeshire & Peterborough        £   37,085         20.4%         24.1%
Hampshire & Solent                   £   36,619         13.6%         20.9%
York & North Yorkshire               £   34,100         28.6%         33.0%
Greater Manchester                   £   34,017         27.8%         32.2%
East Midlands                        £   31,570         20.3%         24.3%
West Yorkshire                       £   30,665         23.9%         26.4%
Sussex & Brighton                    £   29,823         19.3%         23.9%
Cumbria                              £   29,808         19.8%         26.5%
Greater Essex       

In [9]:
# Manual verification for YNYCA
print("\n── Manual YNYCA verification ────────────────────────────────────")
ynyca_itl3 = itl3[itl3["itl_code"].isin(["TLE21","TLE22"])][
    ["itl_code","region_name","2023","pop_2023"]
]
print(ynyca_itl3.to_string())

york_gva  = float(ynyca_itl3[ynyca_itl3["itl_code"]=="TLE21"]["2023"].iloc[0])
ny_gva    = float(ynyca_itl3[ynyca_itl3["itl_code"]=="TLE22"]["2023"].iloc[0])
york_pop  = float(ynyca_itl3[ynyca_itl3["itl_code"]=="TLE21"]["pop_2023"].iloc[0])
ny_pop    = float(ynyca_itl3[ynyca_itl3["itl_code"]=="TLE22"]["pop_2023"].iloc[0])

manual_weighted = (york_gva * york_pop + ny_gva * ny_pop) / (york_pop + ny_pop)
print(f"\n  York GVA/head:           £{york_gva:,.0f}  (pop {york_pop:,.0f})")
print(f"  North Yorks GVA/head:    £{ny_gva:,.0f}  (pop {ny_pop:,.0f})")
print(f"  Manual weighted avg:     £{manual_weighted:,.0f}")
print(f"  Script calculated:       £{msa_df[msa_df['msa_name']=='York & North Yorkshire']['gva_ph_2023'].iloc[0]:,.0f}")
print(f"  Match: {abs(manual_weighted - msa_df[msa_df['msa_name']=='York & North Yorkshire']['gva_ph_2023'].iloc[0]) < 1}")


── Manual YNYCA verification ────────────────────────────────────
   itl_code      region_name     2023  pop_2023
30    TLE21             York  41162.0  206746.0
31    TLE22  North Yorkshire  31778.0  628846.0

  York GVA/head:           £41,162  (pop 206,746)
  North Yorks GVA/head:    £31,778  (pop 628,846)
  Manual weighted avg:     £34,100
  Script calculated:       £34,100
  Match: True


In [11]:
"""
update_excel_gva.py
===================
Writes GVA per head ranks and values into the MSA Indicator Tracker.
Adds methodology caveat to codebook sheet.
"""

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# BASE      = "/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map"
XLSX_PATH = f"{BASE}/data/analysis/MSA_Indicator_Tracker.xlsx"
CSV_PATH  = f"{BASE}/data/processed/msa_gva_ranks.csv"

# ── Load data ──────────────────────────────────────────────────────────────
ranks  = pd.read_csv(CSV_PATH)
wb     = load_workbook(XLSX_PATH)
ws2    = wb["2. YNYCA Profile"]
ws3    = wb["3. MSA Comparison"]
ws4    = wb["4. Codebook"]

ENG_GVA_2023    = 36632
ENG_GVA_2019    = 30514
ENG_GROWTH_1923 = round((ENG_GVA_2023 / ENG_GVA_2019 - 1) * 100, 1)
ENG_GROWTH_1823 = 24.2
GREEN_BG        = "C8E6C9"
DARK            = "1A1A1A"

def write(ws, row, col, value, fmt=None, align="right", bold=False, wrap=False):
    c = ws.cell(row=row, column=col, value=value)
    c.font = Font(name="Arial", size=9, bold=bold, color=DARK)
    c.alignment = Alignment(horizontal=align, vertical="center", wrap_text=wrap)
    if fmt:
        c.number_format = fmt
    return c

def mark_status(ws, row, status="Populated"):
    bg = {"Populated": "C8E6C9", "Partial": "FFF9C4", "To do": "FFCCBC"}.get(status, "FFFFFF")
    c = ws.cell(row=row, column=14, value=status)
    c.fill = PatternFill("solid", fgColor=bg)
    c.font = Font(name="Arial", size=9)
    c.alignment = Alignment(horizontal="center", vertical="center")

def formula(ws, row, col, expr, fmt=None):
    c = ws.cell(row=row, column=col, value=expr)
    c.font = Font(name="Arial", size=9, color=DARK)
    c.alignment = Alignment(horizontal="right", vertical="center")
    if fmt:
        c.number_format = fmt

# ── Pull YNYCA row ─────────────────────────────────────────────────────────
ynyca = ranks[ranks["msa_name"] == "York & North Yorkshire"].iloc[0]

# Leader on the 2018-23 growth measure (computed, not assumed —
# YNYCA leads on 2019-23 but NOT on 2018-23, where it's rank 2 behind
# West of England; narrative text below must reflect whichever MSA
# actually tops gva_growth_2018_23 each run)
leader_1823       = ranks.sort_values("gva_growth_2018_23", ascending=False).iloc[0]
leader_1823_name  = leader_1823["msa_name"]
leader_1823_value = leader_1823["gva_growth_2018_23"]

print(f"YNYCA GVA per head 2023:   £{ynyca['gva_ph_2023']:,.0f}")
print(f"YNYCA GVA per head 2019:   £{ynyca['gva_ph_2019']:,.0f}")
print(f"YNYCA growth 2019-23:       {ynyca['gva_growth_2019_23']:.1f}%")
print(f"YNYCA growth 2018-23:       {ynyca['gva_growth_2018_23']:.1f}%")
print(f"YNYCA England index:        {ynyca['gva_index_eng']:.1f}")
print(f"YNYCA rank GVA level:       {int(ynyca['rank_gva_ph'])}/20")
print(f"YNYCA rank GVA growth 1923: {int(ynyca['rank_gva_growth'])}/20")
print(f"YNYCA rank GVA growth 1823: {int(ynyca['rank_gva_growth_1823'])}/20"
      f"  (leader: {leader_1823_name} at {leader_1823_value:.1f}%)")

# ── Find rows in Sheet 2 ───────────────────────────────────────────────────
id_to_row = {}
for row in ws2.iter_rows(min_row=4):
    v = row[0].value
    if v:
        id_to_row[v] = row[0].row

# ── Update SNAP_004: GVA per head ─────────────────────────────────────────
r = id_to_row.get("SNAP_004")
if r:
    write(ws2, r, 4,  int(ynyca["gva_ph_2023"]),          fmt="£#,##0")
    write(ws2, r, 5,  "£",                                 align="left")
    write(ws2, r, 6,  "2023",                              align="left")
    write(ws2, r, 7,  ENG_GVA_2023,                        fmt="£#,##0")
    formula(ws2, r, 8,  f"=D{r}-G{r}",                    fmt="£#,##0")
    write(ws2, r, 9,  "Higher = better",                   align="left")
    write(ws2, r, 10, int(ynyca["gva_ph_2019"]),           fmt="£#,##0")
    formula(ws2, r, 11, f"=D{r}-J{r}",                    fmt="£#,##0")
    write(ws2, r, 12, ENG_GVA_2023 - ENG_GVA_2019,        fmt="£#,##0")
    write(ws2, r, 13, int(ynyca["rank_gva_ph"]),
          align="center", bold=True)
    write(ws2, r, 15,
          f"ONS Regional GVA (Balanced) Table 2, 2023. "
          f"Pop-weighted avg of ITL3 areas TLE21 (York £41,162) and TLE22 (North Yorks £31,778) "
          f"using MYE 2023 populations. England index: {ynyca['gva_index_eng']:.1f}. "
          f"See Codebook for methodology caveat.",
          align="left")
    mark_status(ws2, r, "Populated")
    print(f"\n  ✓  SNAP_004 updated — row {r}")

# ── Add SNAP_005: GVA growth rate 2018-2023 ───────────────────────────────
# Find next empty row in Sheet 2 after existing indicators
# or find if SNAP_005 already exists
r5 = id_to_row.get("SNAP_005")
if r5:
    write(ws2, r5, 4,  round(ynyca["gva_growth_2018_23"], 1), fmt='0.0"%"')
    write(ws2, r5, 5,  "%",                                    align="left")
    write(ws2, r5, 6,  "2018-2023",                            align="left")
    write(ws2, r5, 7,  ENG_GROWTH_1823,                        fmt='0.0"%"')
    formula(ws2, r5, 8, f"=D{r5}-G{r5}",                      fmt='0.0"%"')
    write(ws2, r5, 9,  "Higher = better",                      align="left")
    write(ws2, r5, 10, round(ynyca["gva_growth_2019_23"], 1),  fmt='0.0"%"')
    write(ws2, r5, 13, int(ynyca["rank_gva_growth_1823"]),
          align="center", bold=True)
    write(ws2, r5, 15,
          f"ONS GVA growth 2018-2023 preferred over 2019-2023 to avoid COVID base distortion. "
          f"YNYCA rank {int(ynyca['rank_gva_growth_1823'])}/20 on 2018-23 measure "
          f"(behind {leader_1823_name} at {leader_1823_value:.1f}%); "
          f"rank {int(ynyca['rank_gva_growth'])}/20 on 2019-23 measure. "
          f"Growth reflects recovery trajectory not starting level — YNYCA still below England avg.",
          align="left")
    mark_status(ws2, r5, "Populated")
    print(f"  ✓  SNAP_005 updated — row {r5}")

# ── Update Sheet 3: all 20 MSAs ────────────────────────────────────────────
print("\n── Updating Sheet 3 MSA Comparison ─────────────────────────────")

NAME_MAP = {
    "Greater Manchester":            "Greater Manchester",
    "West Midlands":                 "West Midlands",
    "North East":                    "North East",
    "Liverpool City Region":         "Liverpool City Region",
    "South Yorkshire":               "South Yorkshire",
    "West Yorkshire":                "West Yorkshire",
    "West of England":               "West of England",
    "Lancashire":                    "Lancashire",
    "Tees Valley":                   "Tees Valley",
    "York & North Yorkshire":        "York & North Yorkshire",
    "Hull & East Yorkshire":         "Hull & East Yorkshire",
    "Greater Lincolnshire":          "Greater Lincolnshire",
    "East Midlands":                 "East Midlands",
    "Cambridgeshire & Pboro":        "Cambridgeshire & Peterborough",
    "Sussex & Brighton":             "Sussex & Brighton",
    "Hampshire & Solent":            "Hampshire & Solent",
    "Cheshire & Warrington":         "Cheshire & Warrington",
    "Cumbria":                       "Cumbria",
    "Greater Essex":                 "Greater Essex",
    "Norfolk & Suffolk":             "Norfolk & Suffolk",
}

# Find MSA column positions in Sheet 3
msa_cols = {}
for col in range(1, 30):
    v = ws3.cell(row=2, column=col).value
    if v:
        parts = str(v).split("\n")
        name = parts[1].strip() if len(parts) >= 2 else parts[0].strip()
        msa_cols[name] = col

# Find indicator rows in Sheet 3
comp_id_rows = {}
for row in ws3.iter_rows(min_row=3):
    v = row[0].value
    if v:
        comp_id_rows[v] = row[0].row

# Find or create SNAP_004 and SNAP_005 rows
snap4_row = comp_id_rows.get("SNAP_004")
snap5_row = comp_id_rows.get("SNAP_005")

max_row = max(comp_id_rows.values()) if comp_id_rows else 3

if not snap4_row:
    snap4_row = max_row + 1
    ws3.cell(row=snap4_row, column=1).value = "SNAP_004"
    ws3.cell(row=snap4_row, column=2).value = "GVA per head (£)"
    ws3.cell(row=snap4_row, column=3).value = "£"
    ws3.cell(row=snap4_row, column=4).value = ENG_GVA_2023
    ws3.cell(row=snap4_row, column=4).number_format = "£#,##0"
    ws3.cell(row=snap4_row, column=4).font = Font(name="Arial", size=9)
    ws3.cell(row=snap4_row, column=4).alignment = Alignment(horizontal="center", vertical="center")
    print(f"  Added SNAP_004 to Sheet 3 row {snap4_row}")
    max_row = snap4_row

if not snap5_row:
    snap5_row = max_row + 1
    ws3.cell(row=snap5_row, column=1).value = "SNAP_005"
    ws3.cell(row=snap5_row, column=2).value = "GVA per head growth 2018-23 (%)"
    ws3.cell(row=snap5_row, column=3).value = "%"
    ws3.cell(row=snap5_row, column=4).value = ENG_GROWTH_1823
    ws3.cell(row=snap5_row, column=4).font = Font(name="Arial", size=9)
    ws3.cell(row=snap5_row, column=4).alignment = Alignment(horizontal="center", vertical="center")
    print(f"  Added SNAP_005 to Sheet 3 row {snap5_row}")

for sheet_name, csv_name in NAME_MAP.items():
    col = msa_cols.get(sheet_name)
    if col is None:
        print(f"  ⚠  Column not found: '{sheet_name}'")
        continue

    row_data = ranks[ranks["msa_name"] == csv_name]
    if row_data.empty:
        print(f"  ⚠  No data for: '{csv_name}'")
        continue

    r = row_data.iloc[0]

    # GVA per head
    if snap4_row:
        c = ws3.cell(row=snap4_row, column=col,
                     value=int(r["gva_ph_2023"]))
        c.number_format = "£#,##0"
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="center", vertical="center")
        # Colour code by England index
        idx = float(r["gva_index_eng"])
        if idx >= 100:
            c.fill = PatternFill("solid", fgColor="C8E6C9")
        elif idx >= 80:
            c.fill = PatternFill("solid", fgColor="FFF9C4")
        else:
            c.fill = PatternFill("solid", fgColor="FFCDD2")

    # GVA growth 2018-23
    if snap5_row:
        c = ws3.cell(row=snap5_row, column=col,
                     value=round(float(r["gva_growth_2018_23"]), 1))
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="center", vertical="center")
        # Colour by vs England growth
        growth = float(r["gva_growth_2018_23"])
        if growth >= ENG_GROWTH_1823:
            c.fill = PatternFill("solid", fgColor="C8E6C9")
        else:
            c.fill = PatternFill("solid", fgColor="FFCDD2")

    print(f"  ✓  {sheet_name:<32}  GVA=£{int(r['gva_ph_2023']):,}  "
          f"idx={float(r['gva_index_eng']):.1f}  "
          f"growth={float(r['gva_growth_2018_23']):.1f}%  "
          f"rank={int(r['rank_gva_ph'])}")

# ── Update Codebook — add GVA methodology note ────────────────────────────
print("\n── Updating Codebook (Sheet 4) ──────────────────────────────────")

# Find next empty row in codebook
cb_rows = {}
for row in ws4.iter_rows(min_row=3):
    v = row[0].value
    if v:
        cb_rows[v] = row[0].row

cb_max = max(cb_rows.values()) if cb_rows else 2

# Check if GVA entry already exists
if "SNAP_004" not in cb_rows:
    r_cb = cb_max + 1
    ws4.row_dimensions[r_cb].height = 80

    cb_data = [
        "SNAP_004",
        "GVA per head (£)",
        "Standard cross-MSA productivity measure. Directly cited in all MSA growth plans "
        "and ONS Subnational Indicators Explorer. Normalises for authority size.",
        "ONS Regional GVA (Balanced) Table 2 (ITL level). Where MSA spans multiple ITL3 "
        "areas, population-weighted average of ITL3 per head figures using ONS MYE 2023 "
        "LA populations aggregated to ITL3 via LAD→ITL3 lookup (Jan 2025). "
        "Boundary check confirmed ITL3 areas match MSA LAD boundaries exactly for all 20 MSAs.",
        "Higher = better (more productive economy)",
        "IMPORTANT: ITL3 GVA figures are ONS estimates via top-down apportionment from higher "
        "geographies — not direct measures at MSA level. Use for comparative ranking and "
        "trajectory analysis, not as precise absolute figures. "
        "Growth: use 2018-2023 not 2019-2023 to avoid COVID base effect distortion — "
        "2019 was the pre-COVID peak and 2020 was anomalously low, making 2019-23 growth "
        "partly a recovery measure rather than genuine economic growth.",
        "Jun 2026",
    ]

    for j, val in enumerate(cb_data, 1):
        c = ws4.cell(row=r_cb, column=j, value=val)
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
        c.fill = PatternFill("solid", fgColor="F5F5F5" if r_cb % 2 == 0 else "FFFFFF")

    # Add SNAP_005 growth codebook entry
    r_cb2 = r_cb + 1
    ws4.row_dimensions[r_cb2].height = 60

    cb_data2 = [
        "SNAP_005",
        "GVA per head growth 2018-23 (%)",
        "Captures economic growth trajectory. Preferred over 2019-23 to avoid COVID distortion. "
        "Key for assessing whether below-average MSAs are converging toward England average.",
        "Derived from SNAP_004 values: (GVA_2023 / GVA_2018 - 1) × 100.",
        "Higher = better (faster growing economy)",
        f"Note that high growth from a low base (e.g. Tees Valley, South Yorkshire) does not "
        f"imply convergence toward England average — check absolute level alongside growth rate. "
        f"YNYCA is rank {int(ynyca['rank_gva_growth_1823'])}/20 on this measure "
        f"(behind {leader_1823_name} at {leader_1823_value:.1f}%) but still 6.9% below "
        f"England average on level (index 93.1).",
        "Jun 2026",
    ]

    for j, val in enumerate(cb_data2, 1):
        c = ws4.cell(row=r_cb2, column=j, value=val)
        c.font = Font(name="Arial", size=9)
        c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
        c.fill = PatternFill("solid", fgColor="FFFFFF")

    print(f"  ✓  Added SNAP_004 codebook entry — row {r_cb}")
    print(f"  ✓  Added SNAP_005 codebook entry — row {r_cb2}")
else:
    print(f"  SNAP_004 already in codebook — row {cb_rows['SNAP_004']}")

# ── Also add to Sources sheet ──────────────────────────────────────────────
ws5 = wb["5. Sources"]

# Find next empty row
src_rows = []
for row in ws5.iter_rows(min_row=3):
    if row[0].value:
        src_rows.append(row[0].row)

src_max = max(src_rows) if src_rows else 2

# Check if already added
existing_src = [ws5.cell(row=r, column=1).value for r in src_rows]
if not any("GVA" in str(v) for v in existing_src if v):
    r_src = src_max + 1
    ws5.row_dimensions[r_src].height = 22

    src_data = [
        "ONS Regional GVA (Balanced) per head — Table 2",
        "ONS",
        "GVA per head, GVA total, GVA growth rates. ITL1/2/3 and CA level.",
        "https://www.ons.gov.uk/economy/grossvalueaddedgva/datasets/nominalregionalgrossvalueaddedbalancedperheadandincomecomponents/current",
        "ITL3 → CA (via LAD→ITL3 lookup)",
    ]

    for j, val in enumerate(src_data, 1):
        c = ws5.cell(row=r_src, column=j, value=val)
        c.font = Font(name="Arial", size=9,
                      color="2A9D8F" if j == 4 else "1A1A1A")
        c.alignment = Alignment(horizontal="left", vertical="center")
        c.fill = PatternFill("solid", fgColor="F5F5F5" if r_src % 2 == 0 else "FFFFFF")

    print(f"  ✓  Added GVA to Sources sheet — row {r_src}")

# ── Save ───────────────────────────────────────────────────────────────────
wb.save(XLSX_PATH)
print(f"\n── Saved: {XLSX_PATH}")
print("\n── YNYCA GVA summary ────────────────────────────────────────────")
print(f"  GVA per head 2023:   £{int(ynyca['gva_ph_2023']):,}  (rank {int(ynyca['rank_gva_ph'])}/20)")
print(f"  England index:        {ynyca['gva_index_eng']:.1f}  (England = 100)")
print(f"  Growth 2018-2023:     {ynyca['gva_growth_2018_23']:.1f}%  (rank {int(ynyca['rank_gva_growth_1823'])}/20, England: {ENG_GROWTH_1823}%)")
print(f"  Story: rank {int(ynyca['rank_gva_growth_1823'])}/20 on 2018-23 trajectory (behind {leader_1823_name}), "
      f"below England average on level")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/ECObsv/Main/Projects/msa_map/data/processed/msa_gva_ranks.csv'

In [ ]:
# ── Refresh SNAP_005 codebook caveat if the row already exists ────────────
# (the block above only WRITES this row on first creation; on a re-run it's
# skipped entirely, which would leave a stale/incorrect caveat in place —
# e.g. the original "YNYCA is rank 1 on growth" text from before this fix)
cb_rows_refresh = {row[0].value: row[0].row for row in ws4.iter_rows(min_row=3) if row[0].value}
r_cb5 = cb_rows_refresh.get("SNAP_005")
if r_cb5:
    caveat = (
        f"Note that high growth from a low base (e.g. Tees Valley, South Yorkshire) does not "
        f"imply convergence toward England average — check absolute level alongside growth rate. "
        f"YNYCA is rank {int(ynyca['rank_gva_growth_1823'])}/20 on this measure "
        f"(behind {leader_1823_name} at {leader_1823_value:.1f}%) but still 6.9% below "
        f"England average on level (index 93.1)."
    )
    c = ws4.cell(row=r_cb5, column=6, value=caveat)
    c.font = Font(name="Arial", size=9)
    c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    wb.save(XLSX_PATH)
    print(f"  ✓  Refreshed SNAP_005 codebook caveat — row {r_cb5}")

  ✓  Refreshed SNAP_005 codebook caveat — row 10


In [12]:
# ── GVA by industry (LAD-level, 1998–2018) ────────────────────────────────────
import pandas as pd, numpy as np, altair as alt, warnings
warnings.filterwarnings('ignore')
alt.data_transformers.enable('default', max_rows=None)

BASE  = '/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Documents/GitHub/RADataHub/msa_map'
NAVY  = '#122b39'; TEAL = '#36b7b4'; AMBER = '#e8a84a'; RED = '#e6224b'; MID = '#aaaaaa'

# Load raw data
_gva_ind = pd.read_csv(f'{BASE}/data/raw/gva/gva-by-industry-by-local-authority-time-series-v1.csv', low_memory=False)
_gva_ind = _gva_ind.rename(columns={
    'v4_1': 'value',
    'administrative-geography': 'lad_code',
    'Time': 'year',
    'sic-unofficial': 'sic',
})
_gva_ind['value'] = pd.to_numeric(_gva_ind['value'], errors='coerce')
_gva_cp = _gva_ind[_gva_ind['Prices'] == 'Current prices'].copy()

# MSA lookup (same pattern as msa_gva_ranks)
ca  = pd.read_csv(f'{BASE}/data/raw/lookups/Local_Authority_District_to_Combined_Authority_(May_2025)_Lookup_in_EN.csv')
dpp = pd.read_csv(f'{BASE}/data/raw/lookups/dpp_lad_lookup_manual.csv')
_lad_msa = pd.concat([
    ca[['LAD25CD', 'CAUTH25NM']].rename(columns={'LAD25CD': 'lad_code', 'CAUTH25NM': 'msa_name'}),
    dpp[['lad25_code', 'dpp_area']].rename(columns={'lad25_code': 'lad_code', 'dpp_area': 'msa_name'}),
]).drop_duplicates('lad_code')
# North Yorkshire: data uses pre-2023 district codes (E07000163–169)
_NY_OLD = [f'E0700016{i}' for i in range(3, 10)]
_lad_msa = pd.concat([_lad_msa, pd.DataFrame({'lad_code': _NY_OLD, 'msa_name': 'York and North Yorkshire'})]).drop_duplicates('lad_code')

_gva_msa = _gva_cp.merge(_lad_msa, on='lad_code', how='left')
_msa_only = _gva_msa[_gva_msa['msa_name'].notna()].copy()

NAME_MAP = {
    'York and North Yorkshire': 'YNYCA',
    'Greater Lincolnshire': 'Greater Lincolnshire',
    'Hull and East Yorkshire': 'Hull & East Yorkshire',
    'Devon and Torbay': 'Devon & Torbay',
    'Norfolk & Suffolk': 'Norfolk & Suffolk',
    'North East': 'North East',
    'Cambridgeshire and Peterborough': 'Cambs & Peterborough',
    'Tees Valley': 'Tees Valley',
    'East Midlands': 'East Midlands',
    'Sussex & Brighton': 'Sussex & Brighton',
    'West Yorkshire': 'West Yorkshire',
    'South Yorkshire': 'South Yorkshire',
    'Lancashire': 'Lancashire',
    'West Midlands': 'West Midlands',
    'Cheshire & Warrington': 'Cheshire & Warrington',
    'Hampshire & Solent': 'Hampshire & Solent',
    'West of England': 'West of England',
    'Greater Essex': 'Greater Essex',
    'Greater Manchester': 'Greater Manchester',
    'Liverpool City Region': 'Liverpool City Region',
}

print(f"GVA-by-industry rows (current prices): {len(_gva_cp):,}")
print(f"MSA-matched rows: {len(_msa_only):,}")
print(f"MSAs matched: {_msa_only['msa_name'].nunique()}")
print(f"Years: {sorted(_msa_only['year'].unique())[:3]} … {sorted(_msa_only['year'].unique())[-3:]}")
print(f"Available ABDE data for YNYCA: {(_msa_only[(_msa_only['msa_name']=='York and North Yorkshire') & (_msa_only['sic']=='ABDE')]).shape[0]} rows")


GVA-by-industry rows (current prices): 465,276
MSA-matched rows: 202,188
MSAs matched: 20
Years: [np.int64(1998), np.int64(1999), np.int64(2000)] … [np.int64(2016), np.int64(2017), np.int64(2018)]
Available ABDE data for YNYCA: 168 rows


In [16]:
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name="article")

In [17]:
# Chart 1 — Cross-MSA ABDE share (Agriculture/mining/energy proxy), 2018
YEAR = 2018
_pivot = _msa_only[_msa_only['sic'].isin(['total', 'ABDE']) & (_msa_only['year'] == YEAR)] \
    .groupby(['msa_name', 'sic'])['value'].sum().reset_index()
_wide = _pivot.pivot_table(index='msa_name', columns='sic', values='value').reset_index()
_wide.columns.name = None
_wide = _wide.rename(columns={'total': 'gva_total', 'ABDE': 'gva_abde'})
_wide['abde_share'] = _wide['gva_abde'] / _wide['gva_total'] * 100
_wide['is_ynyca']   = _wide['msa_name'] == 'York and North Yorkshire'
_wide['short_name'] = _wide['msa_name'].map(NAME_MAP).fillna(_wide['msa_name'])
_wide = _wide.sort_values('abde_share', ascending=True)
MSA_ORDER = _wide['short_name'].tolist()

bars = alt.Chart(_wide).mark_bar(cornerRadiusEnd=3).encode(
    y=alt.Y('short_name:N', sort=MSA_ORDER, title=None,
            axis=alt.Axis(labelFontSize=11, labelColor='#333')),
    x=alt.X('abde_share:Q', title='% of total GVA (current prices, 2018)',
            scale=alt.Scale(domain=[0, 11]),
            axis=alt.Axis(format='d', grid=True, gridColor='#eeeeee', gridOpacity=0.8, tickCount=6)),
    color=alt.condition(alt.datum.is_ynyca, alt.value(TEAL), alt.value('#cccccc')),
    size=alt.condition(alt.datum.is_ynyca, alt.value(18), alt.value(14)),
    tooltip=['msa_name:N', alt.Tooltip('abde_share:Q', format='.1f', title='ABDE share %'),
             alt.Tooltip('gva_abde:Q', format=',.0f', title='ABDE GVA £m'),
             alt.Tooltip('gva_total:Q', format=',.0f', title='Total GVA £m')],
)
labels = alt.Chart(_wide).mark_text(align='left', dx=4, fontSize=10, fontWeight='bold').encode(
    y=alt.Y('short_name:N', sort=MSA_ORDER),
    x=alt.X('abde_share:Q'),
    text=alt.Text('abde_share:Q', format='.1f'),
    color=alt.condition(alt.datum.is_ynyca, alt.value(TEAL), alt.value('#999')),
)

chart_abde_share = (
    alt.layer(bars, labels)
    .properties(
        width=380, height=400,
        title=alt.TitleParams(
            text='Agriculture, mining & energy as share of total GVA',
            subtitle='ABDE sector (SIC A+B+D+E). Current prices, 2018. YNYCA = 4th highest of 20 English MSAs.',
            anchor='start', fontSize=13, subtitleFontSize=10,
            subtitleColor='#666', subtitleFontStyle='italic', color=NAVY,
        )
    )
    .configure_view(strokeWidth=0, fill='white')
    .configure(background='white')
    .configure_axis(grid=False, domainColor='#cccccc', tickColor='#cccccc')
)
chart_abde_share.save(f'{BASE}/data/processed/charts/msa_abde_share.png', scale_factor=3)
chart_abde_share


alt.LayerChart(...)

In [14]:
# Chart 2 — YNYCA sector GVA composition vs all-MSA average, 2018
SECTORS = {
    'ABDE': 'Agriculture/mining/energy',
    'C':    'Manufacturing',
    'F':    'Construction',
    'G':    'Wholesale & retail',
    'H':    'Transport & storage',
    'I':    'Accommodation & food',
    'J':    'Information & comms',
    'K':    'Finance & insurance',
    'L':    'Real estate',
    'M':    'Professional services',
    'N':    'Admin & support',
    'O':    'Public admin',
    'P':    'Education',
    'Q':    'Health & social work',
    'R':    'Arts & recreation',
}

_sec = _msa_only[(_msa_only['sic'].isin(SECTORS)) & (_msa_only['year'] == 2018)] \
    .groupby(['msa_name', 'sic'])['value'].sum().reset_index()
_totals = _msa_only[(_msa_only['sic'] == 'total') & (_msa_only['year'] == 2018)] \
    .groupby('msa_name')['value'].sum().rename('gva_total').reset_index()
_sec = _sec.merge(_totals, on='msa_name')
_sec['share'] = _sec['value'] / _sec['gva_total'] * 100

_msa_avg = _sec.groupby('sic')['share'].mean().reset_index().rename(columns={'share': 'msa_avg'})
_yn_sec  = _sec[_sec['msa_name'] == 'York and North Yorkshire'][['sic', 'share']].copy()
_yn_sec  = _yn_sec.merge(_msa_avg, on='sic')
_yn_sec['sector'] = _yn_sec['sic'].map(SECTORS)
_yn_sec['diff']   = _yn_sec['share'] - _yn_sec['msa_avg']
_yn_sec['above']  = _yn_sec['diff'] > 0
_yn_sec = _yn_sec.sort_values('share', ascending=True)
SEC_ORDER = _yn_sec['sector'].tolist()

connector = alt.Chart(_yn_sec).mark_rule(strokeWidth=1.5, opacity=0.4).encode(
    y=alt.Y('sector:N', sort=SEC_ORDER, title=None,
            axis=alt.Axis(labelFontSize=10.5, labelColor='#333')),
    x=alt.X('msa_avg:Q', scale=alt.Scale(domain=[0, 20]),
            axis=alt.Axis(format='d', grid=True, gridColor='#eeeeee', gridOpacity=0.8,
                          title='% of total GVA (2018)', tickCount=5)),
    x2='share:Q',
    color=alt.condition(alt.datum.above, alt.value(TEAL), alt.value(RED)),
)
dot_avg = alt.Chart(_yn_sec).mark_circle(size=70, opacity=0.5, color='#aaaaaa').encode(
    y=alt.Y('sector:N', sort=SEC_ORDER),
    x=alt.X('msa_avg:Q', scale=alt.Scale(domain=[0, 20])),
    tooltip=[alt.Tooltip('sector:N'), alt.Tooltip('msa_avg:Q', format='.1f', title='MSA average %')],
)
dot_yn = alt.Chart(_yn_sec).mark_circle(size=90, opacity=0.95).encode(
    y=alt.Y('sector:N', sort=SEC_ORDER),
    x=alt.X('share:Q', scale=alt.Scale(domain=[0, 20])),
    color=alt.condition(alt.datum.above, alt.value(TEAL), alt.value(RED)),
    tooltip=[alt.Tooltip('sector:N'), alt.Tooltip('share:Q', format='.1f', title='YNYCA %'),
             alt.Tooltip('diff:Q', format='+.1f', title='vs MSA avg (pp)')],
)

chart_sectors = (
    alt.layer(connector, dot_avg, dot_yn)
    .properties(
        width=380, height=420,
        title=alt.TitleParams(
            text='YNYCA sector GVA composition vs MSA average',
            subtitle='Coloured dot = YNYCA. Grey dot = average of all 20 English MSAs. Teal = YNYCA above average, red = below.',
            anchor='start', fontSize=13, subtitleFontSize=10,
            subtitleColor='#666', subtitleFontStyle='italic', color=NAVY,
        )
    )
    .configure_view(strokeWidth=0, fill='white')
    .configure(background='white')
    .configure_axis(grid=False, domainColor='#cccccc', tickColor='#cccccc')
)
chart_sectors.save(f'{BASE}/data/processed/charts/ynyca_gva_sectors.png', scale_factor=3)
chart_sectors


alt.LayerChart(...)

In [15]:
# Chart 3 — ABDE share over time: YNYCA vs top rural MSAs, 1998–2018
TOP = ['York and North Yorkshire', 'Greater Lincolnshire', 'Hull and East Yorkshire',
       'Devon and Torbay', 'Norfolk & Suffolk']

_trend = _msa_only[_msa_only['sic'].isin(['total', 'ABDE']) & _msa_only['msa_name'].isin(TOP)] \
    .groupby(['msa_name', 'year', 'sic'])['value'].sum().reset_index()
_tw = _trend.pivot_table(index=['msa_name', 'year'], columns='sic', values='value').reset_index()
_tw.columns.name = None
_tw = _tw.rename(columns={'total': 'gva_total', 'ABDE': 'gva_abde'})
_tw['abde_share'] = _tw['gva_abde'] / _tw['gva_total'] * 100
_tw['short_name'] = _tw['msa_name'].map(NAME_MAP).fillna(_tw['msa_name'])
_tw['is_ynyca']   = _tw['msa_name'] == 'York and North Yorkshire'

MSA_C = ['YNYCA', 'Greater Lincolnshire', 'Hull & East Yorkshire', 'Devon & Torbay', 'Norfolk & Suffolk']
MSA_R = [TEAL, '#e8774a', '#888888', '#aaaaaa', '#cccccc']

lines = alt.Chart(_tw).mark_line(point=False).encode(
    x=alt.X('year:O', title='Year',
            axis=alt.Axis(labelFontSize=10, values=[1998, 2002, 2006, 2010, 2014, 2018])),
    y=alt.Y('abde_share:Q', title='ABDE share of total GVA (%)',
            scale=alt.Scale(domain=[0, 13]),
            axis=alt.Axis(format='d', grid=True, gridColor='#eeeeee', gridOpacity=0.8)),
    color=alt.Color('short_name:N',
                    scale=alt.Scale(domain=MSA_C, range=MSA_R),
                    legend=alt.Legend(title=None, orient='top-left', labelFontSize=10, symbolStrokeWidth=2.5)),
    strokeWidth=alt.condition(alt.datum.is_ynyca, alt.value(2.5), alt.value(1.5)),
    opacity=alt.condition(alt.datum.is_ynyca, alt.value(1.0), alt.value(0.65)),
    tooltip=['short_name:N', 'year:O', alt.Tooltip('abde_share:Q', format='.1f', title='ABDE share %')],
)

chart_abde_trend = (
    lines.properties(
        width=380, height=240,
        title=alt.TitleParams(
            text='ABDE share of GVA: top rural MSAs, 1998–2018',
            subtitle='Agriculture, mining, electricity, gas, water & waste (SIC A+B+D+E). Current prices.',
            anchor='start', fontSize=13, subtitleFontSize=10,
            subtitleColor='#666', subtitleFontStyle='italic', color=NAVY,
        )
    )
    .configure_view(strokeWidth=0, fill='white')
    .configure(background='white')
    .configure_axis(grid=False, domainColor='#cccccc', tickColor='#cccccc')
)
chart_abde_trend.save(f'{BASE}/data/processed/charts/msa_abde_trend.png', scale_factor=3)
chart_abde_trend


alt.Chart(...)